# Assignment 11 – CNNs on Tiny ImageNet (3 classes × 200 images)

**Contents**
1. CNN building blocks on a 3×3 matrix (NumPy from scratch)
2. Custom CNN + over/under-fitting analysis (70/15/15 split)
3. Transfer learning: VGG16, MobileNet, ResNet50 vs. custom CNN

> Run in Google Colab (GPU runtime recommended: *Runtime → Change runtime type → GPU*).

## 0. Setup & imports

In [ ]:
import os, glob, random, zipfile, urllib.request
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix)
import pandas as pd

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("TensorFlow:", tf.__version__, "| GPUs:", tf.config.list_physical_devices('GPU'))

---
# Task 1 – CNN basics on a random 3×3 matrix (from scratch with NumPy)

In [ ]:
np.random.seed(1)
X = np.random.randint(0, 10, (3, 3)).astype(float)   # random 3x3 "image"
K = np.random.randn(3, 3).round(2)                   # random 3x3 kernel
print("Input matrix X:\n", X)
print("\nKernel K:\n", K)

### 1.1 Padding (P = 1)

In [ ]:
def pad2d(x, p=1):
    return np.pad(x, pad_width=p, mode='constant', constant_values=0)

Xp = pad2d(X, 1)
print("Padded matrix (5x5):\n", Xp)

### 1.2 Convolution  (output size = (N + 2P − F)/S + 1 = (3 + 2 − 3)/1 + 1 = 3)

In [ ]:
def conv2d(x, k, stride=1):
    n, f = x.shape[0], k.shape[0]
    out = (n - f) // stride + 1
    y = np.zeros((out, out))
    for i in range(out):
        for j in range(out):
            patch = x[i*stride:i*stride+f, j*stride:j*stride+f]
            y[i, j] = np.sum(patch * k)
    return y

conv_out = conv2d(Xp, K)
print("Convolution output (3x3):\n", conv_out.round(3))

### 1.3 Activation functions (ReLU / Sigmoid)

In [ ]:
def relu(x):    return np.maximum(0, x)
def sigmoid(x): return 1 / (1 + np.exp(-x))

relu_out, sig_out = relu(conv_out), sigmoid(conv_out)
print("ReLU:\n", relu_out.round(3))
print("\nSigmoid:\n", sig_out.round(3))

### 1.4 Max pooling, Min pooling, Global Average Pooling

In [ ]:
def pool2d(x, size=2, stride=1, mode='max'):
    out = (x.shape[0] - size) // stride + 1
    y = np.zeros((out, out))
    fn = {'max': np.max, 'min': np.min, 'avg': np.mean}[mode]
    for i in range(out):
        for j in range(out):
            y[i, j] = fn(x[i*stride:i*stride+size, j*stride:j*stride+size])
    return y

def global_avg_pool(x):
    return x.mean()

print("Max pooling (2x2, stride 1):\n", pool2d(relu_out, 2, 1, 'max').round(3))
print("\nMin pooling (2x2, stride 1):\n", pool2d(relu_out, 2, 1, 'min').round(3))
print("\nGlobal Average Pooling:", round(global_avg_pool(relu_out), 4))

### 1.5 Flattening & Fully Connected (Dense) layer

In [ ]:
flat = relu_out.flatten()
print("Flattened vector (9,):", flat.round(3))

def dense(x, W, b):
    return x @ W + b

np.random.seed(3)
W = np.random.randn(9, 4) * 0.5      # 9 inputs -> 4 neurons
b = np.zeros(4)
fc_out = relu(dense(flat, W, b))
print("Dense layer output (4 neurons, ReLU):", fc_out.round(3))

### 1.6 Batch Normalization

In [ ]:
def batch_norm(x, gamma=1.0, beta=0.0, eps=1e-5):
    mean = x.mean(axis=0)
    var  = x.var(axis=0)
    x_hat = (x - mean) / np.sqrt(var + eps)
    return gamma * x_hat + beta

# a mini-batch of 4 feature vectors (each from a random 3x3 matrix -> pad -> conv -> flatten)
np.random.seed(7)
batch = np.stack([conv2d(pad2d(np.random.randint(0, 10, (3, 3)).astype(float)), K).flatten()
                  for _ in range(4)])
bn = batch_norm(batch)
print("Before BN -> mean:", batch.mean(0).round(2)[:4], "std:", batch.std(0).round(2)[:4])
print("After  BN -> mean:", bn.mean(0).round(2)[:4],   "std:", bn.std(0).round(2)[:4])

### 1.7 Dropout (inverted dropout)

In [ ]:
def dropout(x, p=0.5, training=True):
    if not training:
        return x
    mask = (np.random.rand(*x.shape) > p) / (1 - p)   # scale to keep expectation
    return x * mask

np.random.seed(5)
print("Input   :", flat.round(2))
print("Dropout :", dropout(flat, p=0.5).round(2))

### 1.8 Early Stopping (demo on a simulated validation-loss curve)

In [ ]:
class EarlyStoppingDemo:
    def __init__(self, patience=3, min_delta=0.0):
        self.patience, self.min_delta = patience, min_delta
        self.best, self.wait, self.stopped_epoch = np.inf, 0, None
    def step(self, val_loss, epoch):
        if val_loss < self.best - self.min_delta:
            self.best, self.wait = val_loss, 0
        else:
            self.wait += 1
            if self.wait >= self.patience:
                self.stopped_epoch = epoch
                return True
        return False

val_losses = [1.0, 0.8, 0.65, 0.55, 0.52, 0.53, 0.55, 0.58, 0.60, 0.65]
es = EarlyStoppingDemo(patience=3)
for ep, vl in enumerate(val_losses, 1):
    print(f"epoch {ep:2d}  val_loss={vl:.2f}  best={es.best:.2f}  wait={es.wait}")
    if es.step(vl, ep):
        print(f"--> Early stopping at epoch {ep}; best val_loss = {es.best:.2f}")
        break

---
# Data – Tiny ImageNet: 3 classes × 200 images (600 images, 64×64×3)

In [ ]:
URL = "http://cs231n.stanford.edu/tiny-imagenet-200.zip"
ROOT = "tiny-imagenet-200"

if not os.path.exists(ROOT):
    print("Downloading Tiny ImageNet (~240 MB)...")
    urllib.request.urlretrieve(URL, "tiny-imagenet-200.zip")
    with zipfile.ZipFile("tiny-imagenet-200.zip") as z:
        z.extractall(".")
    print("Done.")

In [ ]:
# wnid -> human-readable name
wnid2name = {}
with open(f"{ROOT}/words.txt") as f:
    for line in f:
        wnid, name = line.strip().split("\t")
        wnid2name[wnid] = name

# --- choose any 3 classes (edit this list if you like) ---
all_wnids = sorted(os.listdir(f"{ROOT}/train"))
rng = random.Random(SEED)
CLASSES = rng.sample(all_wnids, 3)
class_names = [wnid2name[w].split(",")[0] for w in CLASSES]
print("Selected classes:", list(zip(CLASSES, class_names)))

IMGS_PER_CLASS = 200
images, labels = [], []
for label, wnid in enumerate(CLASSES):
    paths = sorted(glob.glob(f"{ROOT}/train/{wnid}/images/*.JPEG"))[:IMGS_PER_CLASS]
    for p in paths:
        images.append(np.array(Image.open(p).convert("RGB")))   # some images are grayscale -> force RGB
        labels.append(label)

X_all = np.array(images, dtype=np.uint8)
y_all = np.array(labels)
print("X shape:", X_all.shape, "| y shape:", y_all.shape, "| per-class:", np.bincount(y_all))

In [ ]:
fig, axes = plt.subplots(3, 6, figsize=(12, 6))
for c in range(3):
    idx = np.where(y_all == c)[0][:6]
    for j, i in enumerate(idx):
        axes[c, j].imshow(X_all[i]); axes[c, j].axis("off")
        if j == 0: axes[c, j].set_title(class_names[c], fontsize=10, loc="left")
plt.tight_layout(); plt.show()

### 70% / 15% / 15% stratified split

In [ ]:
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X_all, y_all, test_size=0.30, stratify=y_all, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)

print("Train:", X_train.shape, "| Val:", X_val.shape, "| Test:", X_test.shape)   # 420 / 90 / 90

---
# Task 2 – Custom CNN & over/under-fitting analysis

Architecture (uses every block from Task 1):
`Conv → BatchNorm → ReLU → MaxPool` (×3) → `GlobalAveragePooling` → `Flatten` → `Dense` → `Dropout` → `Softmax`,
trained with **EarlyStopping**.

In [ ]:
def build_custom_cnn(input_shape=(64, 64, 3), n_classes=3, dropout=0.5):
    inp = layers.Input(shape=input_shape)
    x = layers.Rescaling(1./255)(inp)

    for filters in (32, 64, 128):
        x = layers.Conv2D(filters, 3, padding="same")(x)    # convolution + padding
        x = layers.BatchNormalization()(x)                  # batch norm
        x = layers.Activation("relu")(x)                    # activation
        x = layers.MaxPooling2D(2)(x)                       # max pooling

    x = layers.GlobalAveragePooling2D()(x)                  # global average pooling
    x = layers.Flatten()(x)                                 # flatten
    x = layers.Dense(64, activation="relu")(x)              # fully connected
    x = layers.Dropout(dropout)(x)                          # dropout
    out = layers.Dense(n_classes, activation="softmax")(x)
    return models.Model(inp, out, name="custom_cnn")

custom_cnn = build_custom_cnn()
custom_cnn.compile(optimizer=optimizers.Adam(1e-3),
                   loss="sparse_categorical_crossentropy", metrics=["accuracy"])
custom_cnn.summary()

In [ ]:
early_stop = callbacks.EarlyStopping(monitor="val_loss", patience=10,
                                     restore_best_weights=True, verbose=1)

history = custom_cnn.fit(X_train, y_train,
                         validation_data=(X_val, y_val),
                         epochs=80, batch_size=32,
                         callbacks=[early_stop], verbose=2)

### Training vs. validation curves

In [ ]:
def plot_history(h, title):
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
    ax[0].plot(h.history["loss"], label="Train loss")
    ax[0].plot(h.history["val_loss"], label="Validation loss")
    ax[0].axvline(np.argmin(h.history["val_loss"]), ls="--", c="gray", label="Best epoch")
    ax[0].set_title(f"{title} – Loss"); ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Loss"); ax[0].legend()
    ax[1].plot(h.history["accuracy"], label="Train acc")
    ax[1].plot(h.history["val_accuracy"], label="Validation acc")
    ax[1].set_title(f"{title} – Accuracy"); ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("Accuracy"); ax[1].legend()
    plt.tight_layout(); plt.show()

plot_history(history, "Custom CNN")

In [ ]:
def diagnose_fit(h):
    tr_loss, va_loss = h.history["loss"][-1], h.history["val_loss"][-1]
    tr_acc,  va_acc  = h.history["accuracy"][-1], h.history["val_accuracy"][-1]
    best_ep = int(np.argmin(h.history["val_loss"])) + 1
    gap = va_loss - tr_loss
    print(f"Final train loss = {tr_loss:.3f} | val loss = {va_loss:.3f} | gap = {gap:.3f}")
    print(f"Final train acc  = {tr_acc:.3f} | val acc  = {va_acc:.3f} | best epoch = {best_ep}")
    if tr_acc < 0.65 and va_acc < 0.65:
        print("=> UNDERFITTING: both train and validation accuracy are low (model too simple / trained too little).")
    elif tr_acc - va_acc > 0.10 or gap > 0.30:
        print("=> OVERFITTING: training performance is much better than validation "
              "(validation loss rises / stays high while train loss keeps falling).")
    else:
        print("=> GOOD FIT: train and validation curves are close and both performance levels are good.")

diagnose_fit(history)

### Test-set evaluation of the custom CNN

In [ ]:
def evaluate_model(model, X, y, name):
    probs = model.predict(X, verbose=0)
    pred = probs.argmax(1)
    acc = accuracy_score(y, pred)
    p, r, f1, _ = precision_recall_fscore_support(y, pred, average="macro", zero_division=0)
    print(f"\n=== {name} ===")
    print(classification_report(y, pred, target_names=class_names, zero_division=0))
    return {"Model": name, "Accuracy": acc, "Precision": p, "Recall": r, "F1-score": f1}, pred

results = []
res, pred_custom = evaluate_model(custom_cnn, X_test, y_test, "Custom CNN")
results.append(res)

cm = confusion_matrix(y_test, pred_custom)
plt.imshow(cm, cmap="Blues"); plt.colorbar()
plt.xticks(range(3), class_names, rotation=30); plt.yticks(range(3), class_names)
for i in range(3):
    for j in range(3):
        plt.text(j, i, cm[i, j], ha="center", va="center")
plt.title("Custom CNN – Confusion matrix (test)"); plt.show()

### Optional: demonstrate overfitting explicitly
A bigger network *without* BatchNorm/Dropout/EarlyStopping usually overfits the 420 training images,
which makes the curves easy to compare with the regularised model above.

In [ ]:
def build_overfit_cnn():
    inp = layers.Input((64, 64, 3))
    x = layers.Rescaling(1./255)(inp)
    for f in (32, 64, 128, 256):
        x = layers.Conv2D(f, 3, padding="same", activation="relu")(x)
        x = layers.MaxPooling2D(2)(x)
    x = layers.Flatten()(x)
    x = layers.Dense(512, activation="relu")(x)
    out = layers.Dense(3, activation="softmax")(x)
    return models.Model(inp, out, name="overfit_cnn")

overfit_cnn = build_overfit_cnn()
overfit_cnn.compile(optimizer=optimizers.Adam(1e-3),
                    loss="sparse_categorical_crossentropy", metrics=["accuracy"])
h_over = overfit_cnn.fit(X_train, y_train, validation_data=(X_val, y_val),
                         epochs=40, batch_size=32, verbose=0)
plot_history(h_over, "Unregularised CNN")
diagnose_fit(h_over)

---
# Task 3 – Transfer learning: VGG16, MobileNet, ResNet50

Pre-trained ImageNet backbones are **frozen**; only a new classification head
(`GlobalAveragePooling → Dropout → Dense(3)`) is trained. Images are upscaled to 128×128 inside the model
and passed through each network's own `preprocess_input`.

In [ ]:
from tensorflow.keras.applications import VGG16, MobileNet, ResNet50
from tensorflow.keras.applications import vgg16, mobilenet, resnet50

IMG_TL = 128   # upscale 64x64 -> 128x128 for the pre-trained backbones

def build_transfer_model(base_fn, preprocess_fn, name):
    base = base_fn(weights="imagenet", include_top=False, input_shape=(IMG_TL, IMG_TL, 3))
    base.trainable = False
    inp = layers.Input((64, 64, 3))
    x = layers.Resizing(IMG_TL, IMG_TL)(inp)
    x = layers.Lambda(lambda t: preprocess_fn(tf.cast(t, tf.float32)))(x)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(3, activation="softmax")(x)
    m = models.Model(inp, out, name=name)
    m.compile(optimizer=optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

tl_configs = {
    "VGG16":     (VGG16,     vgg16.preprocess_input),
    "MobileNet": (MobileNet, mobilenet.preprocess_input),
    "ResNet50":  (ResNet50,  resnet50.preprocess_input),
}

tl_histories = {}
for name, (fn, prep) in tl_configs.items():
    print(f"\n########## Training {name} ##########")
    model = build_transfer_model(fn, prep, name)
    es = callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
    h = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                  epochs=30, batch_size=32, callbacks=[es], verbose=2)
    tl_histories[name] = h
    res, _ = evaluate_model(model, X_test, y_test, name)
    results.append(res)

In [ ]:
for name, h in tl_histories.items():
    plot_history(h, name)

### Comparison with the custom CNN (test set, macro-averaged metrics)

In [ ]:
df = pd.DataFrame(results).set_index("Model")
display(df.style.format("{:.4f}").highlight_max(axis=0, color="lightgreen"))

ax = df.plot(kind="bar", figsize=(10, 5), ylim=(0, 1.05), rot=0)
ax.set_title("Test performance: Custom CNN vs. Transfer Learning")
ax.set_ylabel("Score"); ax.legend(loc="lower right")
for c in ax.containers:
    ax.bar_label(c, fmt="%.2f", fontsize=8)
plt.tight_layout(); plt.show()

print("Best model by F1-score:", df["F1-score"].idxmax())

## Observations (fill in after running)
* **Custom CNN:** trained from scratch on only 420 images, so it is prone to over-fitting; BatchNorm, Dropout and EarlyStopping reduce the train/validation gap.
* **Transfer learning (VGG16, MobileNet, ResNet50):** pre-trained ImageNet features usually give a much higher accuracy/F1 with far fewer epochs, because the convolutional filters are already good general feature extractors.
* **Trade-offs:** VGG16 is the largest/slowest, MobileNet is the lightest, ResNet50 is deep with residual connections. Compare the table above to decide which gives the best accuracy-versus-cost balance on this dataset.
* Further improvements: data augmentation, fine-tuning the top backbone layers with a small learning rate, more images per class.